# Binary Hate / Non-Hate Vision Classifier

This notebook trains a strong binary video classifier from the preprocessed frame folders produced by `analysis.ipynb`.

Expected inputs:
- `E:/m-hvc/datasets/old-dataset/video_df.csv`
- `E:/m-hvc/datasets/old-dataset/frames_unique/{source_video_id}/*.jpg`

Outputs:
- `checkpoints/best_binary_vision_classifier.pt`
- `reports/binary_vision_metrics.csv`
- `reports/binary_vision_predictions.csv`
- `reports/training_history.csv`
- `reports/confusion_matrix.png`
- `reports/training_curves.png`

In [26]:
from pathlib import Path
from copy import deepcopy
import math
import random
import warnings

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
import timm
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

warnings.filterwarnings("ignore", category=UserWarning)


In [27]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


PROJECT_ROOT = Path(r"E:/m-hvc")
DATASET_DIR = PROJECT_ROOT / "datasets" / "old-dataset"
MANIFEST_PATH = DATASET_DIR / "video_df.csv"
FRAMES_ROOT = DATASET_DIR / "frames_unique"

NOTEBOOK_DIR = PROJECT_ROOT / "multimodal-hvc" / "hate_non_hate_vision_classification"
CHECKPOINT_DIR = NOTEBOOK_DIR / "checkpoints"
REPORT_DIR = NOTEBOOK_DIR / "reports"
CHECKPOINT_PATH = CHECKPOINT_DIR / "best_binary_vision_classifier.pt"
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / "last_binary_vision_classifier.pt"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

CFG = {
    "seed": 42,
    "image_size": 224,
    "num_frames": 32,
    "batch_size": 4,
    "num_workers": 0,
    "epochs": 18,
    "freeze_epochs": 1,
    "warmup_ratio": 0.08,
    "val_size": 0.15,
    "test_size": 0.15,
    "backbone": "convnext_base.fb_in22k_ft_in1k",
    "fallback_backbones": ["tf_efficientnetv2_s.in21k_ft_in1k", "efficientnet_b3.ra2_in1k", "resnet50.a1_in1k"],
    "embed_dim": 512,
    "temporal_layers": 2,
    "temporal_heads": 8,
    "ffn_dim": 1024,
    "dropout": 0.25,
    "head_lr": 4e-4,
    "backbone_lr": 3e-5,
    "weight_decay": 2e-2,
    "grad_clip": 1.0,
    "patience": 5,
    "accum_steps": 4,
    "ema_decay": 0.999,
    "use_weighted_sampler": True,
    "sampler_weight_cap": 4.0,
    "threshold_min": 0.10,
    "threshold_max": 0.90,
    "threshold_steps": 81,
}

set_seed(CFG["seed"])
print(f"device: {DEVICE}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")


device: cuda
gpu: NVIDIA GeForce RTX 4070


## Load Manifest And Check Frame Coverage

In [28]:
def numeric_frame_key(path: Path):
    stem = path.stem.split("_", 1)[0]
    return (0, int(stem)) if stem.isdigit() else (1, path.stem)


def list_frame_paths(frame_dir):
    frame_dir = Path(frame_dir)
    if not frame_dir.exists():
        return []
    frame_paths = [path for path in frame_dir.iterdir() if path.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    return sorted(frame_paths, key=numeric_frame_key)


manifest_df = pd.read_csv(MANIFEST_PATH)
manifest_df["source_video_id"] = manifest_df["source_video_id"].astype(str)
manifest_df["label"] = manifest_df["label"].astype(int)
manifest_df["frame_dir"] = manifest_df.get(
    "frame_dir",
    manifest_df["source_video_id"].map(lambda value: str(FRAMES_ROOT / str(value))),
)
manifest_df["frame_dir"] = manifest_df.apply(
    lambda row: str(FRAMES_ROOT / str(row.source_video_id)) if pd.isna(row.frame_dir) else str(row.frame_dir),
    axis=1,
)
manifest_df["num_available_frames"] = manifest_df["frame_dir"].map(lambda value: len(list_frame_paths(value)))

missing_frames_df = manifest_df[manifest_df["num_available_frames"] <= 0].copy()
missing_frames_df.to_csv(REPORT_DIR / "missing_frame_dirs.csv", index=False)

clip_df = manifest_df[manifest_df["num_available_frames"] > 0].copy().reset_index(drop=True)
assert set(clip_df["label"].unique()).issubset({0, 1})
assert clip_df["source_video_id"].is_unique

print(f"manifest rows: {len(manifest_df):,}")
print(f"trainable clips with frames: {len(clip_df):,}")
print(f"missing frame dirs: {len(missing_frames_df):,}")
display(clip_df["label"].value_counts().rename_axis("label").reset_index(name="count"))
display(clip_df["num_available_frames"].describe().to_frame().T)
clip_df.head()


manifest rows: 3,283
trainable clips with frames: 3,281
missing frame dirs: 2


,label,count
0,0,1948
1,1,1333


,count,mean,std,min,25%,50%,75%,max
num_available_frames,3281.0,46.509296,23.358774,1.0,28.0,64.0,64.0,64.0


,source_video_id,video_file_name,label,label_source,source,transcription,video_path,has_video_file,frame_dir,num_available_frames
0,R_hate_video_099,R_hate_video_099.mp4,1,dataset.csv,old_dataset,,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True,E:\m-hvc\datasets\old-dataset\frames_unique\R_...,64
1,R_hate_video_100,R_hate_video_100.mp4,1,dataset.csv,old_dataset,Parinkar. Hai. Haiya.,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True,E:\m-hvc\datasets\old-dataset\frames_unique\R_...,64
2,R_hate_video_101,R_hate_video_101.mp4,1,dataset.csv,old_dataset,,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True,E:\m-hvc\datasets\old-dataset\frames_unique\R_...,64
3,R_hate_video_102,R_hate_video_102.mp4,1,dataset.csv,old_dataset,,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True,E:\m-hvc\datasets\old-dataset\frames_unique\R_...,64
4,R_hate_video_103,R_hate_video_103.mp4,1,dataset.csv,old_dataset,The darkest.,E:\m-hvc\datasets\old-dataset\video\R_hate_vid...,True,E:\m-hvc\datasets\old-dataset\frames_unique\R_...,64


## Stratified Train / Validation / Test Split

In [29]:
train_val_df, test_df = train_test_split(
    clip_df,
    test_size=CFG["test_size"],
    random_state=CFG["seed"],
    stratify=clip_df["label"],
)

relative_val_size = CFG["val_size"] / (1.0 - CFG["test_size"])
train_df, val_df = train_test_split(
    train_val_df,
    test_size=relative_val_size,
    random_state=CFG["seed"],
    stratify=train_val_df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_summary = pd.concat(
    [
        train_df["label"].value_counts().rename("train"),
        val_df["label"].value_counts().rename("val"),
        test_df["label"].value_counts().rename("test"),
    ],
    axis=1,
).fillna(0).astype(int)

split_df = pd.concat(
    [
        train_df.assign(split="train"),
        val_df.assign(split="val"),
        test_df.assign(split="test"),
    ],
    ignore_index=True,
)
split_df.to_csv(REPORT_DIR / "binary_vision_splits.csv", index=False)

print(f"train: {len(train_df):,}, val: {len(val_df):,}, test: {len(test_df):,}")
display(split_summary)


train: 2,295, val: 493, test: 493


,train,val,test
label,,,
0,1362,293,293
1,933,200,200


## Dataset And Clip Transforms

In [30]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def uniform_indices(num_items, num_frames):
    if num_items <= 1:
        return np.zeros(num_frames, dtype=np.int32)
    indices = np.linspace(0, num_items - 1, num=num_frames)
    return np.clip(np.round(indices).astype(np.int32), 0, num_items - 1)


def random_segment_indices(num_items, num_frames):
    if num_items <= 1:
        return np.zeros(num_frames, dtype=np.int32)
    if num_items <= num_frames:
        base = uniform_indices(num_items, num_frames)
        jitter = np.random.randint(-1, 2, size=num_frames)
        return np.clip(base + jitter, 0, num_items - 1)

    bins = np.linspace(0, num_items, num_frames + 1)
    sampled = []
    for start, end in zip(bins[:-1], bins[1:]):
        low = int(np.floor(start))
        high = max(low + 1, int(np.ceil(end)))
        sampled.append(np.random.randint(low, min(high, num_items)))
    return np.asarray(sampled, dtype=np.int32)


def sample_frame_paths(frame_dir, num_frames, train=False):
    frame_paths = list_frame_paths(frame_dir)
    if not frame_paths:
        raise FileNotFoundError(f"No frames found in {frame_dir}")
    indices = random_segment_indices(len(frame_paths), num_frames) if train else uniform_indices(len(frame_paths), num_frames)
    return [frame_paths[index] for index in indices]


class ClipTransform:
    def __init__(self, image_size=224, train=True):
        self.image_size = image_size
        self.train = train

    def __call__(self, images):
        processed = []

        if self.train:
            crop_i, crop_j, crop_h, crop_w = transforms.RandomResizedCrop.get_params(
                images[0], scale=(0.70, 1.0), ratio=(0.80, 1.20)
            )
            do_flip = random.random() < 0.5
            brightness = 1.0 + random.uniform(-0.25, 0.25)
            contrast = 1.0 + random.uniform(-0.25, 0.25)
            saturation = 1.0 + random.uniform(-0.20, 0.20)
            grayscale = random.random() < 0.05
        else:
            resize_size = int(self.image_size * 1.15)

        for image in images:
            image = image.convert("RGB")
            if self.train:
                image = TF.resized_crop(
                    image,
                    crop_i,
                    crop_j,
                    crop_h,
                    crop_w,
                    (self.image_size, self.image_size),
                    interpolation=InterpolationMode.BILINEAR,
                )
                if do_flip:
                    image = TF.hflip(image)
                image = TF.adjust_brightness(image, brightness)
                image = TF.adjust_contrast(image, contrast)
                image = TF.adjust_saturation(image, saturation)
                if grayscale:
                    image = TF.rgb_to_grayscale(image, num_output_channels=3)
            else:
                image = TF.resize(image, resize_size, interpolation=InterpolationMode.BILINEAR)
                image = TF.center_crop(image, [self.image_size, self.image_size])

            tensor = TF.to_tensor(image)
            tensor = TF.normalize(tensor, IMAGENET_MEAN, IMAGENET_STD)
            processed.append(tensor)

        return torch.stack(processed, dim=0)


class VideoFrameDataset(Dataset):
    def __init__(self, frame_df, num_frames, image_size, train=False):
        self.frame_df = frame_df.reset_index(drop=True)
        self.num_frames = int(num_frames)
        self.train = bool(train)
        self.transform = ClipTransform(image_size=image_size, train=train)

    def __len__(self):
        return len(self.frame_df)

    def __getitem__(self, index):
        row = self.frame_df.iloc[index]
        frame_paths = sample_frame_paths(row.frame_dir, self.num_frames, train=self.train)
        images = []
        for frame_path in frame_paths:
            with Image.open(frame_path) as image:
                images.append(image.convert("RGB"))

        clip = self.transform(images)
        label = torch.tensor(float(row.label), dtype=torch.float32)
        return {
            "clip": clip,
            "label": label,
            "source_video_id": str(row.source_video_id),
        }


## Model

In [31]:
def build_timm_backbone(preferred_backbone, fallback_backbones):
    candidates = [preferred_backbone] + list(fallback_backbones)
    last_error = None

    for backbone_name in candidates:
        try:
            backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0, global_pool="avg")
            feature_dim = int(backbone.num_features)
            print(f"backbone: {backbone_name} | features: {feature_dim} | pretrained: True")
            return backbone_name, backbone, feature_dim, True
        except Exception as exc:
            print(f"Could not load pretrained {backbone_name}: {exc}")
            last_error = exc

    backbone_name = fallback_backbones[-1] if fallback_backbones else preferred_backbone
    backbone = timm.create_model(backbone_name, pretrained=False, num_classes=0, global_pool="avg")
    feature_dim = int(backbone.num_features)
    print(f"backbone: {backbone_name} | features: {feature_dim} | pretrained: False")
    if last_error is not None:
        print(f"last pretrained load error: {last_error}")
    return backbone_name, backbone, feature_dim, False


class TemporalTransformerClassifier(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone_name, self.backbone, feature_dim, self.pretrained = build_timm_backbone(
            cfg["backbone"], cfg["fallback_backbones"]
        )
        embed_dim = int(cfg["embed_dim"])
        self.feature_proj = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(float(cfg["dropout"])),
        )
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, int(cfg["num_frames"]) + 1, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=int(cfg["temporal_heads"]),
            dim_feedforward=int(cfg["ffn_dim"]),
            dropout=float(cfg["dropout"]),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.temporal_encoder = nn.TransformerEncoder(encoder_layer, num_layers=int(cfg["temporal_layers"]))
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(float(cfg["dropout"])),
            nn.Linear(embed_dim, 1),
        )
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, clip):
        batch_size, num_frames, channels, height, width = clip.shape
        frames = clip.reshape(batch_size * num_frames, channels, height, width)
        features = self.backbone(frames)
        features = features.reshape(batch_size, num_frames, -1)
        features = self.feature_proj(features)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        tokens = torch.cat([cls_tokens, features], dim=1)
        tokens = tokens + self.pos_embed[:, : tokens.shape[1], :]
        encoded = self.temporal_encoder(tokens)
        logits = self.head(encoded[:, 0]).squeeze(-1)
        return logits


def set_backbone_trainable(model, trainable):
    for parameter in model.backbone.parameters():
        parameter.requires_grad_(trainable)


model = TemporalTransformerClassifier(CFG).to(DEVICE)
set_backbone_trainable(model, CFG["freeze_epochs"] <= 0)
print(f"trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"total parameters: {sum(p.numel() for p in model.parameters()):,}")


backbone: convnext_base.fb_in22k_ft_in1k | features: 1024 | pretrained: True
trainable parameters: 4,751,361
total parameters: 92,317,825


## Loaders, Loss, Optimizer

In [32]:
train_dataset = VideoFrameDataset(train_df, CFG["num_frames"], CFG["image_size"], train=True)
val_dataset = VideoFrameDataset(val_df, CFG["num_frames"], CFG["image_size"], train=False)
test_dataset = VideoFrameDataset(test_df, CFG["num_frames"], CFG["image_size"], train=False)

train_labels = train_df["label"].to_numpy(dtype=np.int64)
class_counts = np.bincount(train_labels, minlength=2).astype(np.float32)
class_weights = len(train_labels) / np.maximum(class_counts, 1.0)
sample_weights = np.asarray([class_weights[label] for label in train_labels], dtype=np.float32)
sample_weights = np.minimum(sample_weights, CFG["sampler_weight_cap"])

sampler = None
shuffle_train = True
if CFG["use_weighted_sampler"]:
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )
    shuffle_train = False

pin_memory = DEVICE.type == "cuda"
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=shuffle_train,
    sampler=sampler,
    num_workers=CFG["num_workers"],
    pin_memory=pin_memory,
    drop_last=True,
)
val_loader = DataLoader(val_dataset, batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"], pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"], pin_memory=pin_memory)

neg_count = float((train_df["label"] == 0).sum())
pos_count = float((train_df["label"] == 1).sum())
pos_weight = torch.tensor([neg_count / max(pos_count, 1.0)], dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
scaler = GradScaler(enabled=USE_AMP)

print(f"class_counts: non_hate={int(class_counts[0])}, hate={int(class_counts[1])}")
print(f"pos_weight: {float(pos_weight.item()):.4f}")
print(f"train batches: {len(train_loader):,}, val batches: {len(val_loader):,}, test batches: {len(test_loader):,}")


class_counts: non_hate=1362, hate=933
pos_weight: 1.4598
train batches: 573, val batches: 124, test batches: 124


In [33]:
def split_decay(named_params):
    decay = []
    no_decay = []
    for name, parameter in named_params:
        if parameter.ndim <= 1 or name.endswith(".bias") or "norm" in name.lower():
            no_decay.append(parameter)
        else:
            decay.append(parameter)
    return decay, no_decay


def build_optimizer(model):
    backbone_named = []
    head_named = []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("backbone."):
            backbone_named.append((name, parameter))
        else:
            head_named.append((name, parameter))

    param_groups = []
    for named_params, lr in ((backbone_named, CFG["backbone_lr"]), (head_named, CFG["head_lr"])):
        decay, no_decay = split_decay(named_params)
        if decay:
            param_groups.append({"params": decay, "lr": lr, "weight_decay": CFG["weight_decay"]})
        if no_decay:
            param_groups.append({"params": no_decay, "lr": lr, "weight_decay": 0.0})
    if not param_groups:
        raise RuntimeError("No trainable parameters found")
    return torch.optim.AdamW(param_groups, betas=(0.9, 0.999))


def build_scheduler(optimizer, total_steps, warmup_steps, min_lr_ratio=0.08):
    total_steps = max(1, int(total_steps))
    warmup_steps = max(0, int(warmup_steps))

    def lr_lambda(step):
        if warmup_steps > 0 and step < warmup_steps:
            return max(1e-6, float(step + 1) / float(warmup_steps))
        progress = 0.0
        if total_steps > warmup_steps:
            progress = float(step - warmup_steps) / float(total_steps - warmup_steps)
        progress = min(max(progress, 0.0), 1.0)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def create_ema_model(model):
    ema_model = deepcopy(model).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)
    return ema_model


def update_ema(model, ema_model, decay):
    with torch.no_grad():
        model_state = model.state_dict()
        ema_state = ema_model.state_dict()
        for key, ema_value in ema_state.items():
            model_value = model_state[key].detach()
            if torch.is_floating_point(ema_value):
                ema_value.mul_(decay).add_(model_value, alpha=1.0 - decay)
            else:
                ema_value.copy_(model_value)


steps_per_epoch = max(1, math.ceil(len(train_loader) / max(1, CFG["accum_steps"])))
optimizer = build_optimizer(model)
scheduler = build_scheduler(
    optimizer,
    total_steps=steps_per_epoch * max(1, CFG["freeze_epochs"]),
    warmup_steps=max(1, int(steps_per_epoch * max(1, CFG["freeze_epochs"]) * CFG["warmup_ratio"])),
)
ema_model = create_ema_model(model)


## Metrics And Training Helpers

In [34]:
def safe_auc(metric_fn, y_true, y_prob):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(metric_fn(y_true, y_prob))
    except Exception:
        return np.nan


def compute_metrics(y_true, y_prob, threshold=0.5, prefix=""):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= float(threshold)).astype(int)
    return {
        f"{prefix}threshold": float(threshold),
        f"{prefix}accuracy": float(accuracy_score(y_true, y_pred)),
        f"{prefix}f1": float(f1_score(y_true, y_pred, zero_division=0)),
        f"{prefix}precision": float(precision_score(y_true, y_pred, zero_division=0)),
        f"{prefix}recall": float(recall_score(y_true, y_pred, zero_division=0)),
        f"{prefix}roc_auc": safe_auc(roc_auc_score, y_true, y_prob),
        f"{prefix}pr_auc": safe_auc(average_precision_score, y_true, y_prob),
    }


def optimize_threshold(y_true, y_prob):
    thresholds = np.linspace(CFG["threshold_min"], CFG["threshold_max"], CFG["threshold_steps"])
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in thresholds:
        preds = (y_prob >= threshold).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best_f1:
            best_f1 = float(score)
            best_threshold = float(threshold)
    return best_threshold, best_f1


def run_epoch(model, loader, criterion, optimizer=None, scheduler=None, scaler=None, ema_model=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss = 0.0
    total_examples = 0
    all_labels = []
    all_probs = []
    all_ids = []

    if train_mode:
        optimizer.zero_grad(set_to_none=True)

    progress = tqdm(loader, leave=False, desc="train" if train_mode else "eval")
    for step, batch in enumerate(progress, start=1):
        clips = batch["clip"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(clips)
                loss = criterion(logits, labels)
                loss_for_backward = loss / max(1, CFG["accum_steps"])

            if train_mode:
                scaler.scale(loss_for_backward).backward()
                should_step = step % CFG["accum_steps"] == 0 or step == len(loader)
                if should_step:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    if scheduler is not None:
                        scheduler.step()
                    if ema_model is not None:
                        update_ema(model, ema_model, CFG["ema_decay"])

        batch_size = labels.shape[0]
        total_loss += float(loss.detach().cpu().item()) * batch_size
        total_examples += batch_size
        probs = torch.sigmoid(logits.detach()).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.detach().cpu().numpy())
        all_ids.extend(batch["source_video_id"])
        progress.set_postfix(loss=total_loss / max(1, total_examples))

    y_true = np.concatenate(all_labels).astype(int)
    y_prob = np.concatenate(all_probs).astype(float)
    return total_loss / max(1, total_examples), y_true, y_prob, all_ids


def save_checkpoint(path, model, threshold, metrics, epoch):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "config": CFG,
            "backbone_name": model.backbone_name,
            "threshold": float(threshold),
            "metrics": metrics,
            "epoch": int(epoch),
        },
        path,
    )


## Baselines

In [35]:
val_prior = np.full(len(val_df), train_df["label"].mean(), dtype=np.float32)
majority_prob = np.full(len(val_df), 0.0 if train_df["label"].mean() < 0.5 else 1.0, dtype=np.float32)

baseline_rows = []
baseline_rows.append({"baseline": "majority_class", **compute_metrics(val_df["label"].values, majority_prob, 0.5)})
baseline_rows.append({"baseline": "train_prior_prob", **compute_metrics(val_df["label"].values, val_prior, 0.5)})
baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv(REPORT_DIR / "binary_vision_baselines.csv", index=False)
display(baseline_df)


,baseline,threshold,accuracy,f1,precision,recall,roc_auc,pr_auc
0,majority_class,0.5,0.59432,0.0,0.0,0.0,0.5,0.40568
1,train_prior_prob,0.5,0.59432,0.0,0.0,0.0,0.5,0.40568


## Train

In [36]:
best_score = -1.0
best_threshold = 0.5
history = []
epochs_without_improvement = 0

for epoch in range(1, CFG["epochs"] + 1):
    if CFG["freeze_epochs"] > 0 and epoch == CFG["freeze_epochs"] + 1:
        set_backbone_trainable(model, True)
        optimizer = build_optimizer(model)
        remaining_epochs = CFG["epochs"] - epoch + 1
        total_steps = steps_per_epoch * max(1, remaining_epochs)
        warmup_steps = max(1, int(total_steps * CFG["warmup_ratio"]))
        scheduler = build_scheduler(optimizer, total_steps=total_steps, warmup_steps=warmup_steps)
        print(f"epoch {epoch:02d}: backbone unfrozen")

    train_loss, train_y, train_prob, _ = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        ema_model=ema_model,
    )

    eval_model = ema_model if ema_model is not None else model
    val_loss, val_y, val_prob, _ = run_epoch(eval_model, val_loader, criterion)
    threshold, tuned_f1 = optimize_threshold(val_y, val_prob)
    train_metrics = compute_metrics(train_y, train_prob, threshold=0.5, prefix="train_")
    val_default_metrics = compute_metrics(val_y, val_prob, threshold=0.5, prefix="val_default_")
    val_tuned_metrics = compute_metrics(val_y, val_prob, threshold=threshold, prefix="val_tuned_")

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        **train_metrics,
        **val_default_metrics,
        **val_tuned_metrics,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(REPORT_DIR / "training_history.csv", index=False)

    print(
        f"epoch {epoch:02d} | train_loss={train_loss:.4f} | train_acc={train_metrics['train_accuracy']:.4f} | "
        f"train_f1={train_metrics['train_f1']:.4f} | val_loss={val_loss:.4f} | "
        f"val_acc={val_tuned_metrics['val_tuned_accuracy']:.4f} | "
        f"val_f1={val_tuned_metrics['val_tuned_f1']:.4f} | threshold={threshold:.3f}"
    )

    score = val_tuned_metrics["val_tuned_f1"]
    if score > best_score:
        best_score = score
        best_threshold = threshold
        epochs_without_improvement = 0
        save_checkpoint(CHECKPOINT_PATH, eval_model, best_threshold, val_tuned_metrics, epoch)
        print(f"saved best checkpoint: {CHECKPOINT_PATH}")
    else:
        epochs_without_improvement += 1

    save_checkpoint(LAST_CHECKPOINT_PATH, eval_model, threshold, val_tuned_metrics, epoch)

    if epochs_without_improvement >= CFG["patience"]:
        print(f"early stopping after {CFG['patience']} epochs without improvement")
        break

history_df = pd.DataFrame(history)
display(history_df.tail())


epoch 01 | train_loss=0.6721 | train_acc=0.7251 | train_f1=0.7447 | val_loss=0.7330 | val_acc=0.6653 | val_f1=0.6758 | threshold=0.550
saved best checkpoint: E:\m-hvc\multimodal-hvc\hate_non_hate_vision_classification\checkpoints\best_binary_vision_classifier.pt
epoch 02: backbone unfrozen


epoch 02 | train_loss=0.5107 | train_acc=0.8098 | train_f1=0.8233 | val_loss=0.6743 | val_acc=0.6957 | val_f1=0.7126 | threshold=0.470
saved best checkpoint: E:\m-hvc\multimodal-hvc\hate_non_hate_vision_classification\checkpoints\best_binary_vision_classifier.pt


epoch 03 | train_loss=0.4629 | train_acc=0.8298 | train_f1=0.8332 | val_loss=0.6356 | val_acc=0.7323 | val_f1=0.7203 | threshold=0.580
saved best checkpoint: E:\m-hvc\multimodal-hvc\hate_non_hate_vision_classification\checkpoints\best_binary_vision_classifier.pt


epoch 04 | train_loss=0.4082 | train_acc=0.8599 | train_f1=0.8612 | val_loss=0.6129 | val_acc=0.7586 | val_f1=0.7350 | threshold=0.610
saved best checkpoint: E:\m-hvc\multimodal-hvc\hate_non_hate_vision_classification\checkpoints\best_binary_vision_classifier.pt


epoch 05 | train_loss=0.3346 | train_acc=0.9040 | train_f1=0.9076 | val_loss=0.6371 | val_acc=0.7951 | val_f1=0.7349 | threshold=0.810


epoch 06 | train_loss=0.2629 | train_acc=0.9315 | train_f1=0.9316 | val_loss=0.6704 | val_acc=0.7992 | val_f1=0.7374 | threshold=0.850
saved best checkpoint: E:\m-hvc\multimodal-hvc\hate_non_hate_vision_classification\checkpoints\best_binary_vision_classifier.pt


epoch 07 | train_loss=0.3079 | train_acc=0.9241 | train_f1=0.9251 | val_loss=0.7290 | val_acc=0.7769 | val_f1=0.7418 | threshold=0.730
saved best checkpoint: E:\m-hvc\multimodal-hvc\hate_non_hate_vision_classification\checkpoints\best_binary_vision_classifier.pt


epoch 08 | train_loss=0.2209 | train_acc=0.9455 | train_f1=0.9479 | val_loss=0.7866 | val_acc=0.7870 | val_f1=0.7445 | threshold=0.800
saved best checkpoint: E:\m-hvc\multimodal-hvc\hate_non_hate_vision_classification\checkpoints\best_binary_vision_classifier.pt


KeyboardInterrupt: 

## Test Evaluation

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()
best_threshold = float(checkpoint.get("threshold", 0.5))

test_loss, test_y, test_prob, test_ids = run_epoch(model, test_loader, criterion)
test_default_metrics = compute_metrics(test_y, test_prob, threshold=0.5, prefix="test_default_")
test_tuned_metrics = compute_metrics(test_y, test_prob, threshold=best_threshold, prefix="test_tuned_")

metrics_df = pd.DataFrame([
    {"split": "test", "threshold_type": "default", "loss": test_loss, **test_default_metrics},
    {"split": "test", "threshold_type": "val_tuned", "loss": test_loss, **test_tuned_metrics},
])
metrics_df.to_csv(REPORT_DIR / "binary_vision_metrics.csv", index=False)

predictions_df = pd.DataFrame(
    {
        "source_video_id": test_ids,
        "label": test_y.astype(int),
        "prob_hate": test_prob,
        "pred_default": (test_prob >= 0.5).astype(int),
        "pred_tuned": (test_prob >= best_threshold).astype(int),
    }
)
predictions_df.to_csv(REPORT_DIR / "binary_vision_predictions.csv", index=False)

display(metrics_df)
display(predictions_df.head())


## Plots

In [ ]:
history_df = pd.read_csv(REPORT_DIR / "training_history.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_accuracy"], label="train_accuracy")
axes[1].plot(history_df["epoch"], history_df["train_f1"], label="train_f1")
axes[1].plot(history_df["epoch"], history_df["val_tuned_accuracy"], label="val_accuracy")
axes[1].plot(history_df["epoch"], history_df["val_tuned_f1"], label="val_f1")
axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("epoch")
axes[1].legend()
fig.tight_layout()
fig.savefig(REPORT_DIR / "training_curves.png", dpi=160)
plt.show()

cm = confusion_matrix(test_y, (test_prob >= best_threshold).astype(int), labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], labels=["non_hate", "hate"])
ax.set_yticks([0, 1], labels=["non_hate", "hate"])
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title(f"Confusion Matrix @ threshold={best_threshold:.3f}")
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        ax.text(col, row, str(cm[row, col]), ha="center", va="center", color="black")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(REPORT_DIR / "confusion_matrix.png", dpi=160)
plt.show()


## Inference Utility

In [ ]:
def load_trained_classifier(checkpoint_path=CHECKPOINT_PATH):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    inference_cfg = checkpoint.get("config", CFG)
    inference_model = TemporalTransformerClassifier(inference_cfg).to(DEVICE)
    inference_model.load_state_dict(checkpoint["model_state_dict"])
    inference_model.eval()
    threshold = float(checkpoint.get("threshold", 0.5))
    return inference_model, threshold, inference_cfg


def predict_frame_dir(frame_dir, checkpoint_path=CHECKPOINT_PATH):
    inference_model, threshold, inference_cfg = load_trained_classifier(checkpoint_path)
    frame_paths = sample_frame_paths(frame_dir, inference_cfg["num_frames"], train=False)
    images = []
    for frame_path in frame_paths:
        with Image.open(frame_path) as image:
            images.append(image.convert("RGB"))
    clip = ClipTransform(inference_cfg["image_size"], train=False)(images).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        with autocast(device_type=DEVICE.type, enabled=USE_AMP):
            prob = torch.sigmoid(inference_model(clip)).item()
    return {
        "prob_hate": float(prob),
        "threshold": float(threshold),
        "prediction": int(prob >= threshold),
    }
